In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import matplotlib
import scipy as sp
from sklearn.linear_model import LogisticRegression
import util

from scipy.stats import qmc
from numpy.random import Generator, PCG64

import tqdm
from pathlib import Path

# TE-TE Simulations

## Pre-processing

In [9]:
%run DataProcessingTE.py
# Outputs the following:
    # df_AM22 - processed individual TE data from Armstrong McKay et al 2022
    # df_W24 - TE-TE interactions from Wunderling et al 2024
    # df_SSPs - global mean surface temp. anomalies vs pre-industrial (1850-1900). Historical and several SSPs
        # See scenarios for the list of scenarios 

betas = []
value_at_min=0.001
for x in df_AM22[['Threshold (°C) - Min.','Threshold (°C) - Est.','Threshold (°C) - Max.']].values:
    betas.append(float(sigmoid_beta(x[0],x[1],x[2],value_at_min=value_at_min)))
df_AM22['Sigmoid Beta'] = betas
Tmin = df_AM22.loc['ENSO']['Threshold (°C) - Min.']
Tmax = df_AM22.loc['ENSO']['Threshold (°C) - Max.']
df_AM22.loc['ENSO','Sigmoid Beta'] = np.log(1/value_at_min - 1)/(np.mean([Tmin,Tmax])-Tmin)
valmin_str = str(value_at_min).replace('.','')

/home/jbara/Documents/Code/test-env/Notebooks/DataProcessingTE.py:21: RuntimeWarning: Mean of empty slice
  return np.exp(np.nanmean(np.log([minstr,maxstr])))


In [10]:
df_AM22['Sigmoid Beta']

AMAZ     4.604503
AMOC     2.656444
ASI      3.837086
ENSO     4.604503
GIS      9.866793
PERM     6.906755
REEF    13.813510
WAIS    13.813510
WAM      8.633443
Name: Sigmoid Beta, dtype: float64

## Parameter inference and sampling

In [11]:
seed = 45643564852856852352111
sampler = qmc.LatinHypercube(d=len(df_W24),rng=Generator(PCG64(seed)))
sample = sampler.random(n=500) # still in a [0,1]^18 hypercube 500 samples!

A = []
for r in range(sample.shape[0]):
    G = util.generate_TE_graph(df_W24,sample[r,:],df_AM22)
    A.append(nx.to_numpy_array(G, nodelist=TEs, weight='weight'))
    # np.savetxt("../Data/Simulations/TE/Interaction matrices/A_%s.csv"%r,A[r])

## Simulations under Historical/SSP warming

In [14]:
T_lim = np.array(df_AM22['T_lim'].to_list())
beta = df_AM22['Sigmoid Beta'].to_numpy()
GMST_hist = df_SSPs['Historical']['Mean'].to_numpy()
Nico_time = False

years = np.arange(1950, 2100) 
T = len(years) # number of time steps, each step is 1 year
# probabilities are in year^-1 units
p_c = np.ones((len(scenarios)-1,T,len(df_AM22)))
p_c *= 1/(df_AM22['CollapseTime'].to_numpy()+1)
for t in range(len(GMST_hist)):
    p_c[:,t,:] *= logistic(GMST_hist[t],T_lim,beta)
for t in range(len(GMST_hist),T):
    for i,s in enumerate(scenarios[1:]):
        p_c[i,t,:] *= logistic(df_SSPs[s]['Mean'].to_numpy()[t-len(GMST_hist)],T_lim,beta)

if Nico_time:
    p_r = 1/(df_AM22['RecoveryTime (Nico)'].to_numpy()+1)
else:
    p_r = 1/(df_AM22['RecoveryTime'].to_numpy()+1)

In [15]:
D = np.zeros((len(scenarios)-1,len(sample),len(TEs),T))
in_deg = np.zeros((len(sample),len(TEs)))
# tlevels = np.zeros_like(in_deg)
for r in tqdm.tqdm(range(sample.shape[0])):
    # G = util.generate_TE_graph(df_W24,sample[r,:],df_AM22)
    # A = nx.to_numpy_array(G, nodelist=TEs, weight='weight')
    # tlevels[r,:] = [nx.trophic_levels(G,weight='weight')[n] for n in TEs]
    # in_deg[r,:] = A.sum(axis=0)
    for j in range(len(scenarios)-1):
        for t in range(1,T):
            D[j,r,:,t] = util.environmental_update(D[j,r,:,t-1],A[r],p_c=p_c[j,t,:],p_r=p_r)

D_noint = np.zeros((len(scenarios)-1,len(TEs),T))
for t in tqdm.tqdm(range(1,T)):
    for j in range(len(scenarios)-1):
        D_noint[j,:,t] = util.environmental_update(D_noint[j,:,t-1],np.zeros((len(TEs),len(TEs))),p_c=p_c[j,t,:],p_r=p_r)

if Nico_time:
    folder = '../Data/Simulations/TE/Sigmoid/'+valmin_str+'_at_Tmin_NWrtimes/SSP Warming/'
else:
    folder = '../Data/Simulations/TE/Sigmoid/'+valmin_str+'_at_Tmin_WBrtimes/SSP Warming/'
Path(folder[:-1]).mkdir(parents=True, exist_ok=True)
# Saving the data
for i,te in enumerate(TEs):
    np.savetxt(folder+"D_"+te+"_Historical.csv", D[0,:,i,:len(GMST_hist)])
    np.savetxt(folder+"D_no_interaction_"+te+"_Historical.csv", D_noint[0,i,:len(GMST_hist)])
    for j in range(len(scenarios)-1):
        np.savetxt(folder+"D_"+te+"_"+scenarios[j+1]+".csv", D[j,:,i,len(GMST_hist):])
        np.savetxt(folder+"D_no_interaction_"+te+"_"+scenarios[j+1]+".csv", D_noint[j,i,len(GMST_hist):])

100%|██████████████████████████████████████| 149/149 [00:00<00:00, 15298.31it/s]


## Stationary Distributions under Fixed Warming

## 1.6C

In [16]:
# Current GMST anomaly vs pre-industrial level is given from the EU's Copernicus Climate Change Service (C3S) = 1.6C
    # see https://climate.copernicus.eu/global-climate-highlights-2024
current_GMST = +1.60
T_lim = np.array(df_AM22['T_lim'].to_list())
beta = df_AM22['Sigmoid Beta'].to_numpy()
T = 150 # years
Nico_time = False

p_c = np.ones(len(df_AM22))
p_c *= 1/(df_AM22['CollapseTime'].to_numpy()+1)
p_c *= util.logistic(current_GMST,T_lim,beta)

if Nico_time:
    p_r = 1/(df_AM22['RecoveryTime (Nico)'].to_numpy()+1)
    folder = "../Data/Simulations/TE/Sigmoid/"+valmin_str+"_at_Tmin_NWrtimes/Fixed 1_6/"
else:
    p_r = 1/(df_AM22['RecoveryTime'].to_numpy()+1)
    folder = "../Data/Simulations/TE/Sigmoid/"+valmin_str+"_at_Tmin_WBrtimes/Fixed 1_6/"
Path(folder[:-1]).mkdir(parents=True, exist_ok=True)

In [17]:
def stat_dist_theory_res(D,*data):
    A,q_c,q_r = data
    x_pos = np.dot(np.transpose(np.where(A>=0,A,0)),D)
    x_neg = np.dot(np.transpose(np.where(A<0,-A,0)),D)
    # out = np.zeros(len(D))
    # for i in range(len(D)):
    #     out[i] = D[i] - (1+x_pos[i])*q_c[i]/((1+x_pos)*q_c[i] + (1+x_neg)*q_r[i])
    out = D - (1+x_pos)*q_c/((1+x_pos)*q_c + (1+x_neg)*q_r)
    return out

In [26]:
D_theory = np.array([sp.optimize.fsolve(stat_dist_theory_res, 
                                        np.ones(9), 
                                        args=(A[r],p_c,p_r)) 
                     for r in range(500)])

In [27]:
np.savetxt(folder+"stat_dist.csv",D_theory)
np.savetxt(folder+"stat_dist_no_interactions.csv",p_c/(p_c+p_r))

## Global warming 0 - 10

In [3]:
from scipy.signal import argrelextrema
%run DataProcessingTE.py
# Outputs the following:
    # df_AM22 - processed individual TE data from Armstrong McKay et al 2022
    # df_W24 - TE-TE interactions from Wunderling et al 2024
    # df_SSPs - global mean surface temp. anomalies vs pre-industrial (1850-1900). Historical and several SSPs
        # See scenarios for the list of scenarios 

betas = []
value_at_min=0.001
for x in df_AM22[['Threshold (°C) - Min.','Threshold (°C) - Est.','Threshold (°C) - Max.']].values:
    betas.append(float(sigmoid_beta(x[0],x[1],x[2],value_at_min=value_at_min)))
df_AM22['Sigmoid Beta'] = betas
Tmin = df_AM22.loc['ENSO']['Threshold (°C) - Min.']
Tmax = df_AM22.loc['ENSO']['Threshold (°C) - Max.']
df_AM22.loc['ENSO','Sigmoid Beta'] = np.log(1/value_at_min - 1)/(np.mean([Tmin,Tmax])-Tmin)
# betas=df_AM22['Sigmoid Beta']
valmin_str = str(value_at_min).replace('.','')

/home/jbara/Documents/Code/test-env/Notebooks/DataProcessingTE.py:21: RuntimeWarning: Mean of empty slice
  return np.exp(np.nanmean(np.log([minstr,maxstr])))


In [4]:
def stat_dist_theory_res(D,*data):
    A,q_c,q_r = data
    x_pos = np.dot(np.transpose(np.where(A>=0,A,0)),D)
    x_neg = np.dot(np.transpose(np.where(A<0,-A,0)),D)
    # out = np.zeros(len(D))
    # for i in range(len(D)):
    #     out[i] = D[i] - (1+x_pos[i])*q_c[i]/((1+x_pos)*q_c[i] + (1+x_neg)*q_r[i])
    out = D - (1+x_pos)*q_c/((1+x_pos)*q_c + (1+x_neg)*q_r)
    return out

In [7]:
beta

AMAZ     4.604503
AMOC     2.656444
ASI      3.837086
ENSO     4.604503
GIS      9.866793
PERM     6.906755
REEF    13.813510
WAIS    13.813510
WAM      8.633443
Name: Sigmoid Beta, dtype: float64

In [8]:
A = [np.loadtxt("../Data/Simulations/TE/Interaction matrices/A_%s.csv"%r) for r in range(500)]
q_c = 1/(df_AM22['CollapseTime'].to_numpy()+1)
T_lim = np.array(df_AM22['T_lim'].to_list())
beta=df_AM22['Sigmoid Beta']

Nico_time = False
if Nico_time:
    p_r = 1/(df_AM22['RecoveryTime (Nico)'].to_numpy()+1)
    folder = "../Data/Simulations/TE/Sigmoid/"+valmin_str+"_at_Tmin_NWrtimes/Stat dist over GMST/"
else:
    p_r = 1/(df_AM22['RecoveryTime'].to_numpy()+1)
    folder = "../Data/Simulations/TE/Sigmoid/"+valmin_str+"_at_Tmin_WBrtimes/Stat dist over GMST/"
Path(folder[:-1]).mkdir(parents=True, exist_ok=True)

In [9]:
T = np.arange(0,10,step=0.001)
D_theory = np.array([[sp.optimize.fsolve(stat_dist_theory_res, 
                                        np.ones(9), 
                                        args=(A[r],util.logistic(t, T_lim,beta)*q_c,p_r)) 
                     for t in T] for r in tqdm.tqdm(range(500))])

D_theory_noint = np.array([sp.optimize.fsolve(stat_dist_theory_res, 
                                              np.ones(9), 
                                              args=(np.zeros(9),util.logistic(t, T_lim,beta)*q_c,p_r)) 
                           for t in tqdm.tqdm(T)])

  0%|                                                   | 0/500 [00:15<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
# Saving the data
for i,te in enumerate(TEs):
    np.savetxt(folder+"D_"+te+".csv", D_theory[:,:,i])
    np.savetxt(folder+"D_no_interaction_"+te+".csv", D_theory_noint[:,i])

In [ ]:
D_hot_theory = np.array([sp.optimize.fsolve(stat_dist_theory_res, 
                                            np.ones(9), 
                                            args=(A[r],q_c,p_r))
                         for r in tqdm.tqdm(range(500))])

D_hot_theory_noint = sp.optimize.fsolve(stat_dist_theory_res, 
                                    np.ones(9), 
                                    args=(np.zeros(9),q_c,p_r))

In [34]:
# Saving the data
np.savetxt(folder+"D_hot_allTEs.csv", D_hot_theory)
np.savetxt(folder+"D_hot_no_interaction_allTEs.csv", D_hot_theory_noint)

In [1]:
df_AM22.head()

NameError: name 'df_AM22' is not defined